<a href="https://colab.research.google.com/github/05satyam/AI-ML/blob/feature%2Fpytorch/Distributed_Training_in_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Resources

### **Mental Map: Distributed Training in PyTorch**

**1. Why Go Distributed? (The 'Why?')**

*   **Need for Speed:** Training large models or on massive datasets takes too long on a single device.
*   **Memory Constraints:** Model or batch size might exceed the memory of a single GPU.
*   **Scalability:** Efficiently leverage multiple GPUs or machines.

**2. Core Concepts (The 'Language')**

*   **Process Group:** The communication channel established between all participating processes/nodes. (Think of it as setting up a conference call).
*   **Rank:** A unique identifier for each process within the process group (e.g., 0, 1, 2...). (Each participant in the call has a unique ID).
*   **World Size:** The total number of processes participating in the distributed training. (Total number of participants in the call).
*   **Backend:** The communication library used (e.g., `NCCL` for NVIDIA GPUs, `Gloo` for CPU/multi-CPU, `MPI`).
*   **Node:** A physical machine.
*   **GPU:** Graphics Processing Unit (the workhorse).

**3. Types of Parallelism (The 'How?')**

   *   **A. Data Parallelism (Most Common)**
        *   **Goal:** Distribute data, replicate the *entire* model on each device.
        *   **How it works:** Each device gets a different chunk of data, computes gradients locally, and then gradients are *averaged* across all devices to update *all* model replicas synchronously.
        *   **PyTorch Implementations:**
            *   **`torch.nn.DataParallel` (DP):** Simpler to use (`model = nn.DataParallel(model)`). Less efficient, uses one main GPU for aggregation (potential bottleneck). *Generally discouraged for serious DDP.*
            *   **`torch.nn.parallel.DistributedDataParallel` (DDP):** **THE RECOMMENDED APPROACH.** More efficient, creates separate processes for each device, performs gradient synchronization in a more distributed and optimized way (`all-reduce`). Scales much better.

   *   **B. Model Parallelism (For Huge Models)**
        *   **Goal:** Distribute parts of a *single* model across multiple devices.
        *   **How it works:** Different layers of the model reside on different devices. Data flows sequentially through devices. (Think of an assembly line where each worker does one step).
        *   **When to use:** Model is too large to fit on a single GPU.
        *   **Complexity:** More complex to implement, requires careful partitioning, can suffer from idle devices if layers aren't balanced.

   *   **C. Hybrid Parallelism (Advanced)**
        *   **Goal:** Combine Data and Model Parallelism.
        *   **How it works:** For example, use DDP across multiple nodes, and within each node, use Model Parallelism for very large models. (Imagine multiple assembly lines, and each line is distributed).

**4. Topologies & Strategies (The 'Where?')**

   *   **Single-Node, Multi-GPU:**
        *   All GPUs are on the same machine.
        *   Often uses `NCCL` backend for fast inter-GPU communication.
        *   DDP is the primary strategy.
   *   **Multi-Node, Multi-GPU:**
        *   GPUs spread across several machines (e.g., a cluster).
        *   Requires network configuration (master address, port).
        *   Still uses DDP, but communication also happens over the network between nodes.

**5. Key PyTorch Components (The 'Tools')**

*   **`torch.distributed`:** The fundamental package for distributed communication primitives.
*   **`DistributedDataParallel`:** The wrapper for your model for data parallelism.
*   **`DistributedSampler`:** Essential for `DataLoader` to ensure each DDP process gets a unique subset of data without overlap.
*   **`torch.nn.SyncBatchNorm`:** If your model has Batch Normalization layers, use this to synchronize statistics across all devices, especially with small per-GPU batch sizes, for better training stability.
*   **Launchers (`torchrun` / `torch.distributed.launch`):** Scripts to spawn and manage multiple processes, setting up environment variables (`RANK`, `WORLD_SIZE`, etc.) automatically.

**Your Decision Flow:**

1.  **Is my model or batch too big for one GPU, or training too slow?**
    *   **Yes:** Go Distributed.
    *   **No:** Stick to single-device training.

2.  **Is my *entire model* too big for a single GPU?**
    *   **Yes:** Consider Model Parallelism or Hybrid Parallelism. (More complex!)
    *   **No (only data/batch size is too big, or just want speedup):** Use Data Parallelism.

3.  **For Data Parallelism, which one?**
    *   **Always prefer `DistributedDataParallel` (DDP).** Avoid `DataParallel` unless you have specific reasons (e.g., a very old codebase you can't refactor).


## Key Steps for DDP:

- Initialize the process group: Use dist.init_process_group() to set up communication among all processes.
- Wrap your model: Encapsulate your nn.Module with torch.nn.parallel.DistributedDataParallel.
- Prepare data loaders: Ensure each process gets a unique subset of the data, typically using DistributedSampler with your DataLoader.
- Synchronize batch normalization (if needed): For models with BatchNorm layers, you might need torch.nn.SyncBatchNorm to ensure correct statistics across devices.

### Step 1: Import necessary libraries

In [ ]:
import os
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

### Step 2: Initialize the Distributed Environment

This is crucial for setting up communication between processes. `init_process_group` ensures that all processes can communicate. For a real distributed setup, `MASTER_ADDR`, `MASTER_PORT`, `RANK`, and `WORLD_SIZE` environment variables would be set by the launcher.

In a Colab environment, for a single-machine multi-GPU setup, you might manually set these or use `torch.multiprocessing.spawn` (which is more advanced for this conceptual example). For simplicity, this example assumes these are set by an external launcher, but we'll include placeholders.

`rank` refers to the unique identifier of the current process, and `world_size` is the total number of processes.

In [ ]:
def setup(rank, world_size):
    # These environment variables would typically be set by a launcher like torchrun
    os.environ['MASTER_ADDR'] = os.getenv('MASTER_ADDR', 'localhost')
    os.environ['MASTER_PORT'] = os.getenv('MASTER_PORT', '12355')
    os.environ['RANK'] = str(rank)
    os.environ['WORLD_SIZE'] = str(world_size)

    # Initialize the process group
    dist.init_process_group("nccl" if torch.cuda.is_available() else "gloo", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank) # Set the current device

def cleanup():
    dist.destroy_process_group()

# For demonstration, we'll assume world_size=1 if not in a distributed environment
# In a real DDP setup, these would be managed by the launcher.
# For now, let's define them if they are not already set.
_world_size = int(os.getenv("WORLD_SIZE", "1"))
_rank = int(os.getenv("RANK", "0"))

if _world_size > 1:
    setup(_rank, _world_size)
    print(f"Process {_rank} initialized with world size {_world_size}")
else:
    print("Running in a single-process, non-distributed mode.")

Running in a single-process, non-distributed mode.


### Step 3: Define a simple model

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.linear = nn.Linear(10, 1)

    def forward(self, x):
        return self.linear(x)

### Step 4: Prepare the Dataset and DataLoader

For DDP, it's crucial to use `DistributedSampler` with your `DataLoader`. This sampler ensures that each process receives a unique and non-overlapping subset of the dataset, preventing redundant computations and ensuring correct gradient aggregation.

In [ ]:
class DummyDataset(Dataset):
    def __init__(self, num_samples=100):
        self.data = torch.randn(num_samples, 10)
        self.labels = torch.randn(num_samples, 1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Create a dummy dataset
dataset = DummyDataset(num_samples=1000)

# Create a DistributedSampler
# If not in distributed mode, DistributedSampler will behave like a regular sampler.
# world_size is 1 if not distributed.
# rank is 0 if not distributed.
sampler = DistributedSampler(dataset, num_replicas=_world_size, rank=_rank, shuffle=True)

# Create DataLoader with the sampler
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, sampler=sampler)

print(f"Process {_rank}: Dataset size = {len(dataset)}, Samples for this process = {len(dataloader) * batch_size}")

Process 0: Dataset size = 1000, Samples for this process = 1024


### Step 5: Wrap the model with `DistributedDataParallel`

This step replicates the model on each GPU (for `world_size` > 1) and handles the synchronization of gradients. `device_ids` is used to specify which GPU this particular process should use. `output_device` usually matches `device_ids[0]`.

In [ ]:
device = torch.device(f'cuda:{_rank}' if torch.cuda.is_available() and _world_size > 1 else 'cpu')
model = SimpleModel().to(device)

if _world_size > 1:
    ddp_model = nn.parallel.DistributedDataParallel(model, device_ids=[_rank], output_device=_rank)
    print(f"Process {_rank}: Model wrapped with DistributedDataParallel on device {device}")
else:
    ddp_model = model
    print(f"Process {_rank}: Running model on {device} (not DDP)")

Process 0: Running model on cpu (not DDP)


### Step 6: Define Loss Function and Optimizer

In [ ]:
loss_fn = nn.MSELoss()
optimizer = optim.SGD(ddp_model.parameters(), lr=0.001)

### Step 7: The Training Loop

Each process performs its own forward and backward pass on its subset of data. `DistributedDataParallel` automatically handles the `all-reduce` operation for gradients in the background after `loss.backward()` is called, ensuring that `optimizer.step()` updates the model with averaged gradients across all processes.

In [ ]:
epochs = 2
for epoch in range(epochs):
    if _world_size > 1:
        sampler.set_epoch(epoch) # Essential for shuffling across epochs in DDP

    total_loss = 0.0
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = ddp_model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    if _rank == 0: # Only print from rank 0 to avoid duplicate output
        print(f"Epoch {epoch+1}, Process {_rank}: Average Loss = {avg_loss:.4f}")

Epoch 1, Process 0: Average Loss = 1.1773
Epoch 2, Process 0: Average Loss = 1.1458


### Step 8: Cleanup

Destroy the process group to release resources.

In [ ]:
if _world_size > 1:
    cleanup()
    print(f"Process {_rank}: Cleaned up distributed environment.")

```mermaid
graph TD
    subgraph Launcher (e.g., torchrun)
        L[Spawns Processes] --> P1(Process 0 / GPU 0)
        L --> P2(Process 1 / GPU 1)
        L --> Pn(Process N / GPU N)
    end

    subgraph Each Process (e.g., P1, P2, Pn)
        P1_A[1. init_process_group()] -- Establishes Comm. --> PG(Process Group)
        P1_B[2. Create Dataset] --> P1_C[3. DistributedSampler]
        P1_C --> P1_D[4. DataLoader (local data batch)]
        P1_E[5. Local Model Replica] --> P1_F[6. DistributedDataParallel (DDP)]
        P1_D -- Provides Data --> P1_F
    end

    subgraph Training Loop
        Iter(Iteration Start) --> P1_FL[Forward Pass (local batch)]
        P1_FL --> P1_L[Compute Local Loss]
        P1_L --> P1_BL[Backward Pass (local gradients)]
        P1_BL -- Gradients (all-reduce) --> Sync(Synchronize Gradients via Process Group)
        Sync -- Averaged Gradients --> P1_OS[Optimizer Step (update local model)]
        P1_OS -- Updates --> P1_F
        P1_F -- Updated Local Model --> Iter
    end

    subgraph End of Training
        P_Cleanup[cleanup()] -- Releases Resources --> PG
    end

    style L fill:#f9f,stroke:#333,stroke-width:2px
    style PG fill:#afa,stroke:#333,stroke-width:2px
    style Sync fill:#ffc,stroke:#333,stroke-width:2px
    classDef process_node fill:#cbe8f8,stroke:#333,stroke-width:1px;
    class P1,P2,Pn process_node;
    class P1_A,P1_B,P1_C,P1_D,P1_E,P1_F,P1_FL,P1_L,P1_BL,P1_OS,P_Cleanup process_node;

```